<div style="background:#0d3349;padding:36px 40px 28px;border-radius:10px;color:#fff;font-family:'Segoe UI',sans-serif;">
  <p style="margin:0 0 4px;font-size:12px;letter-spacing:2px;opacity:.6;text-transform:uppercase;">Data Engineering Pipeline</p>
  <h1 style="margin:0 0 10px;font-size:26px;font-weight:700;line-height:1.3;">Automated Data Cleaning Pipeline<br>SYNOP Surface Observation Records</h1>
  <p style="margin:0 0 20px;font-size:13px;opacity:.75;">Stasiun Meteorologi Fatmawati Bengkulu — WMO Station ID 96253</p>
  <hr style="border:none;border-top:1px solid rgba(255,255,255,.2);margin:0 0 18px;"/>
  <table style="font-size:12px;opacity:.8;border-collapse:collapse;">
    <tr><td style="padding:2px 24px 2px 0;">Data Source</td><td>BMKG — Stasiun Meteorologi Kelas I Fatmawati Bengkulu</td></tr>
    <tr><td style="padding:2px 24px 2px 0;">Coverage</td><td>January 2022 – July 2024 &nbsp;|&nbsp; Hourly Resolution</td></tr>
    <tr><td style="padding:2px 24px 2px 0;">Output</td><td>clean_hourly.csv &nbsp;·&nbsp; clean_daily.csv</td></tr>
    <tr><td style="padding:2px 24px 2px 0;">Standard</td><td>WMO-No. 306 Manual on Codes &nbsp;·&nbsp; WMO-No. 8 CIMO Guide</td></tr>
  </table>
</div>


## Deskripsi

Notebook ini mendokumentasikan proses pembersihan dan transformasi data pengamatan permukaan sinoptik (*SYNOP*) dari Stasiun Meteorologi Fatmawati Bengkulu. Data diperoleh dalam format laporan SYNOP mentah yang dikodekan sesuai standar **WMO-No. 306 Manual on Codes** (WMO, 2019) dan perlu melalui serangkaian tahapan pengolahan sebelum dapat digunakan dalam analisis klimatologi maupun visualisasi dashboard.

Pipeline ini menghasilkan dua tabel keluaran dengan granularitas berbeda:

| Keluaran | Granularitas | Baris | Kolom | Keterangan |
|---|---|---|---|---|
| `clean_hourly.csv` | Per jam | ~22.144 | 27 | Seluruh variabel pengamatan per jam yang telah didekode dan divalidasi |
| `clean_daily.csv` | Harian | ~923 | 22 | Agregasi harian beserta variabel instrumen yang hanya dicatat sekali per hari |

**Alur pipeline:**

```
RAW_SYNOP_REPORT.csv
        │
        ├─ 1. Parse timestamp (UTC → datetime)
        ├─ 2. Decode curah hujan (SYNOP IR indicator + kode 8888)
        ├─ 3. Interpolasi angin — metode sirkular (Mardia & Jupp, 2000)
        ├─ 4. Interpolasi linear variabel dengan missing < 2%
        ├─ 5. Quality Control — validasi batas fisik
        ├─ 6. Seleksi & penamaan kolom
        │
        ├──► clean_hourly.csv
        └──► clean_daily.csv (agregasi + ekstraksi variabel harian)
```


---
## Landasan Teori dan Definisi Variabel

Bagian ini mendokumentasikan definisi ilmiah, alat pengukur, dan rumus kalkulasi untuk setiap variabel meteorologi yang digunakan dalam dataset. Referensi utama mengacu pada *Guide to Meteorological Instruments and Methods of Observation* (WMO-No. 8, edisi 2018), yang merupakan standar internasional pengamatan meteorologi permukaan.


### 1. Suhu Udara (*Air Temperature*)

**Definisi.**  
Suhu udara adalah ukuran energi kinetik rata-rata molekul-molekul gas di atmosfer, yang mencerminkan derajat panas atau dinginnya udara. Dalam pengamatan sinoptik, suhu udara permukaan diukur pada ketinggian standar 1,25–2,0 m di atas permukaan tanah, terlindung dari radiasi langsung dan hujan, menggunakan *Stevenson screen* (sangkar meteorologi) (WMO, 2018, §1.3).

**Instrumen.**  
Pengamatan manual menggunakan **termometer merkuri bola kering** (*dry-bulb thermometer*), **termometer bola basah** (*wet-bulb thermometer*), dan **termometer maksimum–minimum**. Termometer bola basah digunakan bersama bola kering untuk menghitung kelembaban relatif melalui psikrometri (WMO, 2018, §4.2).

| Kolom | Instrumen | Satuan |
|---|---|---|
| `suhu_bola_kering_c` | Termometer bola kering | °C |
| `suhu_bola_basah_c` | Termometer bola basah | °C |
| `suhu_titik_embun_c` | Diturunkan dari psikrometri | °C |
| `suhu_max_tercatat_c` | Termometer maksimum | °C |
| `suhu_min_tercatat_c` | Termometer minimum | °C |

**Rumus Suhu Titik Embun** (Magnus approximation — Lawrence, 2005):

$$T_d = \frac{b \cdot \gamma(T, RH)}{a - \gamma(T, RH)}, \quad \text{di mana} \quad \gamma(T, RH) = \frac{a \cdot T}{b + T} + \ln\left(\frac{RH}{100}\right)$$

dengan $a = 17{,}625$, $b = 243{,}04\,^\circ\text{C}$. Persamaan ini memiliki kesalahan di bawah 0,1°C untuk rentang $-40\,^\circ\text{C}$ hingga $60\,^\circ\text{C}$ (Lawrence, 2005).

> **Referensi:** Lawrence, M. G. (2005). The relationship between relative humidity and the dewpoint temperature in moist air: A simple conversion and applications. *Bulletin of the American Meteorological Society*, 86(2), 225–233. https://doi.org/10.1175/BAMS-86-2-225


### 2. Kelembaban Relatif (*Relative Humidity*)

**Definisi.**  
Kelembaban relatif (RH) adalah perbandingan antara tekanan uap air aktual ($e$) dengan tekanan uap jenuh ($e_s$) pada suhu dan tekanan yang sama, dinyatakan dalam persen (WMO, 2018, §4.1):

$$RH = \frac{e}{e_s(T)} \times 100\%$$

Tekanan uap jenuh $e_s$ dihitung menggunakan persamaan **Tetens** yang dimodifikasi oleh Buck (1981):

$$e_s(T) = 6{,}1121 \exp\left(\frac{(18{,}678 - T/234{,}5) \cdot T}{257{,}14 + T}\right) \quad [\text{hPa}]$$

**Instrumen.**  
Pada stasiun manual BMKG, RH diperoleh dari selisih suhu bola kering dan bola basah menggunakan **tabel psikrometri** atau **Psychrometer Assmann**. Stasiun modern menggunakan sensor kapasitif higrometer.

> **Referensi:** Buck, A. L. (1981). New equations for computing vapor pressure and enhancement factor. *Journal of Applied Meteorology*, 20(12), 1527–1532. https://doi.org/10.1175/1520-0450(1981)020


### 3. Tekanan Udara (*Atmospheric Pressure*)

**Definisi.**  
Tekanan udara adalah gaya yang diberikan oleh kolom atmosfer per satuan luas. Dataset ini menyediakan dua representasi tekanan (WMO, 2018, §3.1):

| Kolom | Simbol | Definisi |
|---|---|---|
| `tekanan_qfe_mb` | QFE | Tekanan aktual di permukaan stasiun (*station pressure*) |
| `tekanan_qff_mb` | QFF | Tekanan dikoreksi ke muka laut rata-rata menggunakan formula hipsometrik |

**Rumus reduksi tekanan ke muka laut (QFF):**

$$P_{QFF} = P_{QFE} \cdot \exp\left(\frac{g \cdot z}{R_d \cdot T_{v,m}}\right)$$

di mana $g = 9{,}80665\,\text{m/s}^2$ (percepatan gravitasi), $z$ = ketinggian stasiun (m dpl), $R_d = 287{,}058\,\text{J/(kg·K)}$ (konstanta gas udara kering), dan $T_{v,m}$ adalah suhu virtual rata-rata kolom udara antara stasiun dan muka laut.

**Instrumen.**  
**Barometer Fortin** (merkuri) atau **barometer aneroid digital** (Vaisala PTB220).

> **Referensi:** WMO. (2018). *Guide to meteorological instruments and methods of observation* (WMO-No. 8, 2018 ed.). World Meteorological Organization. https://library.wmo.int/records/item/41650-guide-to-meteorological-instruments-and-methods-of-observation


### 4. Angin (*Wind*)

**Definisi.**  
Angin permukaan diukur sebagai kecepatan rata-rata dan arah datang selama interval 10 menit sebelum waktu pengamatan, pada ketinggian standar 10 m (WMO, 2018, §5.1).

| Kolom | Satuan | Definisi |
|---|---|---|
| `wind_speed_ms` | m/s | Kecepatan angin rata-rata 10 menit |
| `wind_dir_deg` | ° | Arah angin — dari mana angin berasal (0°/360° = Utara, 90° = Timur) |

**Instrumen.** **Anemometer cup** dan **wind vane** (penunjuk arah), dipasang pada tiang 10 m sesuai standar WMO.

**Interpolasi Sirkular untuk Arah Angin.**  
Arah angin merupakan variabel sirkular (nilai $0°$ dan $360°$ identik) sehingga interpolasi linear konvensional menghasilkan estimasi yang tidak valid secara fisika. Misalnya, rata-rata linear dari $350°$ dan $10°$ menghasilkan $180°$ (arah selatan), padahal kedua pengamatan menunjukkan angin dari utara. Metode yang digunakan mengikuti **Mardia & Jupp (2000)** dengan dekomposisi ke komponen Cartesian:

$$\bar{x}_{\sin} = \frac{1}{n}\sum_{i=1}^{n} \sin(\theta_i), \qquad \bar{x}_{\cos} = \frac{1}{n}\sum_{i=1}^{n} \cos(\theta_i)$$

Interpolasi dilakukan pada komponen $\sin(\theta)$ dan $\cos(\theta)$ secara terpisah, kemudian dikembalikan ke sudut dengan fungsi $\text{atan2}$:

$$\theta_{\text{interpolated}} = \text{atan2}(\bar{x}_{\sin,\text{interp}},\ \bar{x}_{\cos,\text{interp}}) \mod 360°$$

> **Referensi:** Mardia, K. V., & Jupp, P. E. (2000). *Directional statistics*. John Wiley & Sons. https://doi.org/10.1002/9780470316979 *(Wiley, Scopus-indexed)*


### 5. Curah Hujan (*Precipitation*) dan Indikator SYNOP IR

**Definisi.**  
Curah hujan adalah jumlah air yang jatuh ke permukaan dalam satuan kedalaman (mm), diasumsikan tidak mengalami penguapan, infiltrasi, atau limpasan (WMO, 2018, §6.1).

**Instrumen.** **Ombrometer (rain gauge) tipe Hellmann** untuk pengukuran harian, dan **penakar hujan tipping-bucket** untuk pengukuran per jam.

**Skema Pengkodean SYNOP — Indikator IR.**  
Dalam laporan SYNOP (WMO-No. 306), kolom `RAINFALL LAST MM` tidak dapat dibaca langsung tanpa merujuk pada `RAINFALL INDICATOR IR`. Nilai IR mendefinisikan periode akumulasi sekaligus validitas pengukuran:

| Nilai IR | Interpretasi | Penanganan |
|---|---|---|
| 0 | Curah hujan diukur, periode 6 jam | Gunakan nilai numerik |
| 1 | Curah hujan diukur, periode 12 jam | Gunakan nilai numerik |
| 2 | Curah hujan diukur, periode 24 jam | Gunakan nilai numerik |
| 3 | Curah hujan diukur, periode 1 jam | Gunakan nilai numerik |
| 4 | Tidak ada hujan / tidak diukur | Set 0.0 mm |

Nilai **8888** dalam kolom curah hujan adalah kode SYNOP untuk *trace* (jejak hujan yang tidak terukur secara kuantitatif) atau pengukuran yang dihilangkan (*omitted*). Nilai ini **bukan nol** dan harus diperlakukan sebagai `NaN`.

> **Referensi:** WMO. (2019). *Manual on codes — International codes, Volume I.1* (WMO-No. 306). World Meteorological Organization. https://library.wmo.int/records/item/35713-manual-on-codes-volume-i-1


### 6. Perawanan (*Cloud Cover and Type*)

**Tutupan Awan (N).**  
Tutupan awan dinyatakan dalam satuan **oktas** (oktan), yaitu perkiraan visual berapa delapan bagian langit yang tertutup awan. Skala 0–9 digunakan dalam SYNOP (WMO, 2018, §15.1):

| Nilai | Makna |
|---|---|
| 0 | Langit cerah (*clear sky*) |
| 1–7 | Tingkat tutupan bertahap |
| 8 | Langit tertutup penuh (*overcast*) |
| 9 | Langit tidak terlihat (kabut, hujan lebat, dsb.) |

**Tipe Awan.**  
Klasifikasi awan mengacu pada **Atlas Awan WMO** (*International Cloud Atlas*, WMO-No. 407) yang membagi awan ke dalam tiga lapisan berdasarkan ketinggian dasar awan:

| Kolom | Lapisan | Ketinggian tipikal | Genus |
|---|---|---|---|
| `tipe_awan_rendah` (CL) | Rendah | < 2.000 m | Cu, Cb, Sc, St, Ns |
| `tipe_awan_menengah` (CM) | Menengah | 2.000–6.000 m | Ac, As |
| `tipe_awan_tinggi` (CH) | Tinggi | > 6.000 m | Ci, Cc, Cs |

> **Referensi:** WMO. (2017). *International cloud atlas: Manual on the observation of clouds and other meteors* (WMO-No. 407). World Meteorological Organization. https://cloudatlas.wmo.int


### 7. Lama Penyinaran Matahari (*Sunshine Duration*)

**Definisi.**  
Lama penyinaran matahari (*sunshine duration*) didefinisikan sebagai periode dalam sehari di mana intensitas radiasi matahari langsung melebihi ambang batas **120 W/m²**, yang secara empiris berhubungan dengan kondisi langit cerah tanpa awan tebal (WMO, 2018, §8.1).

**Instrumen.** **Solarimeter Campbell-Stokes** — bola kaca yang memfokuskan sinar matahari ke pita kertas khusus. Bagian pita yang terbakar merekam lama penyinaran efektif. Nilai dicatat sekali sehari (pukul 00:00 UTC) sebagai akumulasi hari sebelumnya.

**Catatan untuk pipeline ini:** Kolom `SUNSHINE H SSS` dalam data raw hanya muncul pada baris jam 00:00 UTC, dengan persentase *apparent missing* ~95,8%. Ini **bukan data hilang**, melainkan sifat pencatatan harian. Dalam `clean_daily.csv`, nilai ini diekstrak langsung dari baris jam 00:00.

> **Referensi:** Sanchez-Lorenzo, A., Calbó, J., & Wild, M. (2013). Global and diffuse solar radiation in Spain: Building a homogeneous dataset changes and trends. *Global and Planetary Change*, 100, 343–352. https://doi.org/10.1016/j.gloplacha.2012.11.010 *(Elsevier, Scopus Q1)*


### 8. Kode Cuaca Sekarang (*Present Weather — ww*)

**Definisi.**  
Kode `ww` (0–99) dalam SYNOP merepresentasikan kondisi cuaca yang sedang terjadi atau yang telah terjadi dalam satu jam terakhir di stasiun, termasuk fenomena seperti hujan, kabut, badai petir, dan salju. Tabel kode lengkap terdapat dalam WMO-No. 306, Table 4677.

| Rentang Kode | Kategori |
|---|---|
| 00–19 | Tidak ada presipitasi, kabut, badai petir, atau fenomena lain saat pengamatan |
| 20–29 | Presipitasi, kabut, badai petir terjadi dalam 1 jam terakhir |
| 30–39 | Badai pasir, debu, salju, kabut beku |
| 40–49 | Kabut (*fog*) dan kabut beku (*ice fog*) |
| 50–59 | Gerimis (*drizzle*) |
| 60–69 | Hujan (*rain*) |
| 70–79 | Hujan salju dan presipitasi padat |
| 80–99 | Hujan lebat / shower dan badai petir (*thunderstorm*) |

> **Referensi:** WMO. (2019). *Manual on codes — Volume I.1* (WMO-No. 306), Table 4677. World Meteorological Organization.


---
## 1. Dependensi

In [ ]:
!pip install openpyxl -q


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)


## 2. Konfigurasi Lingkungan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
RAW_FILE_PATH = '/content/drive/MyDrive/BMKG/RAW_SYNOP_REPORT.csv'
OUTPUT_DIR    = '/content/drive/MyDrive/BMKG/output'

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

HOURLY_OUT = os.path.join(OUTPUT_DIR, 'clean_hourly.csv')
DAILY_OUT  = os.path.join(OUTPUT_DIR, 'clean_daily.csv')


## 3. Pemuatan Data

In [ ]:
df_raw = pd.read_csv(RAW_FILE_PATH, low_memory=False)

print(f'Dimensi      : {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom')
print(f'Rentang data : {df_raw["DATA TIMESTAMP"].iloc[0]}')
print(f'             : {df_raw["DATA TIMESTAMP"].iloc[-1]}')


In [ ]:
df_raw.head(3)


## 4. Eksplorasi Awal

Inspeksi persentase *missing values* per kolom untuk menentukan strategi penanganan. Variabel dengan *missing* yang tampak tinggi (>90%) perlu diverifikasi terlebih dahulu sebelum diasumsikan sebagai data hilang — beberapa di antaranya merupakan variabel harian yang memang hanya dicatat sekali per hari (§5.7).


In [ ]:
miss_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
pd.DataFrame({'missing_%': miss_pct.round(1)})[miss_pct > 0]


## 5. Tahapan Pembersihan Data

### 5.1 Parsing Timestamp

Kolom `DATA TIMESTAMP` tersimpan dalam format string dengan sufiks timezone (`+0:00`) yang perlu dilepas sebelum konversi ke objek `datetime`. Seluruh catatan menggunakan waktu **UTC** sesuai konvensi SYNOP (WMO-No. 306, §2.2).


In [ ]:
df = df_raw.copy()

df['timestamp'] = pd.to_datetime(
    df['DATA TIMESTAMP'].str.replace(r'\s+\+.*', '', regex=True)
)

df['date']  = df['timestamp'].dt.date
df['hour']  = df['timestamp'].dt.hour
df['year']  = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day']   = df['timestamp'].dt.day

df = df.sort_values('timestamp').reset_index(drop=True)

ts = df['timestamp'].sort_values().reset_index(drop=True)
gaps = ts.diff().dropna()

print(f'Periode data : {df["timestamp"].min()} — {df["timestamp"].max()}')
print(f'Hari unik    : {df["date"].nunique()}')
print(f'Gap (>1 jam) : {(gaps > pd.Timedelta("1h")).sum()} kejadian')


### 5.2 Dekode Curah Hujan

Nilai kolom `RAINFALL LAST MM` ditentukan maknanya oleh `RAINFALL INDICATOR IR` sesuai WMO-No. 306. Nilai **8888** bukan curah hujan — melainkan kode *trace/omitted* yang harus dikonversi ke `NaN`, bukan ke nol (lihat §5 Landasan Teori).


In [ ]:
def decode_rainfall(row):
    ir  = row['RAINFALL INDICATOR IR']
    val = row['RAINFALL LAST MM']

    if ir == 4:
        return 0.0
    if pd.notna(val) and val == 8888:
        return np.nan
    if ir == 3:
        return val if pd.notna(val) else 0.0
    if ir in [0, 1, 2]:
        return val if pd.notna(val) else np.nan
    return np.nan


df['rainfall_mm'] = df.apply(decode_rainfall, axis=1)

print('Distribusi hasil dekode:')
print(f'  Nol (tidak hujan)  : {(df["rainfall_mm"] == 0).sum():>7,}')
print(f'  Terukur (> 0 mm)   : {(df["rainfall_mm"] > 0).sum():>7,}')
print(f'  NaN (trace/omitted): {df["rainfall_mm"].isna().sum():>7,}')
print(f'  Nilai maksimum     : {df["rainfall_mm"].max():.1f} mm')


### 5.3 Interpolasi Arah Angin (Metode Sirkular)

Missing values pada arah angin sebesar 0,1% ditangani menggunakan dekomposisi komponen $\sin/\cos$ mengikuti Mardia & Jupp (2000), sebagaimana dijabarkan pada §4 Landasan Teori.


In [ ]:
def circular_interpolate(series):
    rad = np.deg2rad(series)
    sin_i = pd.Series(np.sin(rad)).interpolate(method='linear', limit_direction='both')
    cos_i = pd.Series(np.cos(rad)).interpolate(method='linear', limit_direction='both')
    return np.rad2deg(np.arctan2(sin_i, cos_i)) % 360


df['wind_dir_deg']  = circular_interpolate(df['WIND DIR DEG DD'])
df['wind_speed_ms'] = df['WIND SPEED FF'].interpolate(method='linear', limit_direction='both')

print(f'Arah angin  — range: {df["wind_dir_deg"].min():.1f}° – {df["wind_dir_deg"].max():.1f}°  '
      f'| missing: {df["wind_dir_deg"].isna().sum()}')
print(f'Kecepatan   — range: {df["wind_speed_ms"].min():.1f} – {df["wind_speed_ms"].max():.1f} m/s '
      f'| missing: {df["wind_speed_ms"].isna().sum()}')


### 5.4 Interpolasi Linear — Variabel dengan *Missing* < 2%

Variabel jarak pandang dan tinggi dasar awan memiliki *missing rate* di bawah 2%, sehingga interpolasi linear temporal aman diterapkan tanpa mengubah karakteristik distribusi data secara signifikan (Moritz & Bartz-Beielstein, 2017).

> Moritz, S., & Bartz-Beielstein, T. (2017). imputeTS: Time series missing value imputation in R. *The R Journal*, 9(1), 207–218. https://doi.org/10.32614/RJ-2017-009 *(CRAN, Scopus-indexed)*


In [ ]:
interp_map = {
    'VISIBILITY VV' : 'visibility_km',
    'CLOUD BASE M H': 'cloud_base_m',
}

for src, dst in interp_map.items():
    df[dst] = df[src].interpolate(method='linear', limit_direction='both')
    n_after = df[dst].isna().sum()
    print(f'{dst:<22} missing setelah interpolasi: {n_after}')


### 5.5 Penamaan Ulang dan Seleksi Kolom

Kolom diberi nama deskriptif menggunakan konvensi *snake_case* disertai satuan, mengikuti panduan *tidy data* (Wickham, 2014).

> Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, 59(10), 1–23. https://doi.org/10.18637/jss.v059.i10 *(Scopus Q1)*


In [ ]:
rename_map = {
    'TEMP DRYBULB C TTTTTT'      : 'suhu_bola_kering_c',
    'TEMP DEWPOINT C TDTDTD'     : 'suhu_titik_embun_c',
    'TEMP WETBULB C'             : 'suhu_bola_basah_c',
    'RELATIVE HUMIDITY PC'       : 'kelembaban_relatif_pct',
    'PRESSURE QFF MB DERIVED'    : 'tekanan_qff_mb',
    'PRESSURE QFE MB DERIVED'    : 'tekanan_qfe_mb',
    'PRESSURE TEND 3H MB PPP'    : 'tendensi_tekanan_3h_mb',
    'CLOUD COVER OKTAS M'        : 'tutupan_awan_oktas',
    'CLOUD LOW TYPE CL'          : 'tipe_awan_rendah',
    'CLOUD MED TYPE CM'          : 'tipe_awan_menengah',
    'CLOUD HIGH TYPE CH'         : 'tipe_awan_tinggi',
    'CLOUD LAYER 2 HEIGHT M HSHS': 'lapisan_awan2_tinggi_m',
    'CLOUD LAYER 2 AMT OKTAS NS' : 'lapisan_awan2_oktas',
    'PRESENT WEATHER WW'         : 'cuaca_sekarang_ww',
    'PAST WEATHER W1'            : 'cuaca_lalu_w1',
}

df = df.rename(columns=rename_map)

HOURLY_COLS = [
    'timestamp', 'year', 'month', 'day', 'hour',
    'suhu_bola_kering_c', 'suhu_titik_embun_c', 'suhu_bola_basah_c',
    'kelembaban_relatif_pct', 'tekanan_qff_mb', 'tekanan_qfe_mb', 'tendensi_tekanan_3h_mb',
    'wind_dir_deg', 'wind_speed_ms',
    'rainfall_mm',
    'visibility_km', 'cloud_base_m', 'tutupan_awan_oktas',
    'tipe_awan_rendah', 'tipe_awan_menengah', 'tipe_awan_tinggi',
    'lapisan_awan2_tinggi_m', 'lapisan_awan2_oktas',
    'cuaca_sekarang_ww', 'cuaca_lalu_w1',
]

df_hourly = df[HOURLY_COLS].copy()
print(f'Tabel per jam: {df_hourly.shape[0]:,} baris × {df_hourly.shape[1]} kolom')


### 5.6 Kendali Kualitas (*Quality Control*)

Validasi batas fisik (*gross error check*) diterapkan sesuai ambang batas klimatologi untuk wilayah tropis Bengkulu. Nilai di luar rentang dikonversi ke `NaN` dan baris yang bersangkutan diberi flag `SUSPECT`. Pendekatan ini mengikuti prosedur *Basic Quality Control* WMO (Zahumenský, 2004).

> Zahumenský, I. (2004). *Guidelines on quality control procedures for data from automatic weather stations*. World Meteorological Organization, CIMO. https://community.wmo.int/en/activity-areas/imop/cimo-publications


In [ ]:
QC_BOUNDS = {
    'suhu_bola_kering_c'    : (15.0,  42.0),
    'suhu_titik_embun_c'    : (10.0,  32.0),
    'suhu_bola_basah_c'     : (15.0,  35.0),
    'kelembaban_relatif_pct': (20.0, 100.0),
    'tekanan_qff_mb'        : (990.0, 1030.0),
    'tekanan_qfe_mb'        : (990.0, 1020.0),
    'wind_speed_ms'         : (0.0,   50.0),
    'wind_dir_deg'          : (0.0,  360.0),
    'rainfall_mm'           : (0.0,  300.0),
    'visibility_km'         : (0.0,   80.0),
    'tutupan_awan_oktas'    : (0.0,    9.0),
}

df_hourly['qc_flag'] = 'OK'

for col, (lo, hi) in QC_BOUNDS.items():
    mask = df_hourly[col].notna() & ((df_hourly[col] < lo) | (df_hourly[col] > hi))
    if mask.sum() > 0:
        df_hourly.loc[mask, 'qc_flag'] = 'SUSPECT'
        df_hourly.loc[mask, col] = np.nan
        print(f'[SUSPECT] {col:<32} {mask.sum()} nilai → NaN')

print()
print(df_hourly['qc_flag'].value_counts().to_string())


### 5.7 Konstruksi Tabel Harian

Tabel harian dikonstruksi dari dua sumber berbeda. Pertama, **agregasi statistik** dari tabel per jam (mean, max, min, sum). Kedua, **ekstraksi langsung** dari variabel yang hanya dicatat sekali sehari oleh instrumen:

- **Tmax / Tmin** — dicatat pada jam **12:00 UTC** (19:00 WIB), merepresentasikan suhu maksimum/minimum periode siang
- **Sunshine / Rainfall 24H** — dicatat pada jam **00:00 UTC** (07:00 WIB), merepresentasikan akumulasi hari sebelumnya

Kedua variabel harian tersebut **tidak boleh diimputasi atau diinterpolasi** pada tabel per jam, karena nilainya bukan pengamatan per jam melainkan akumulasi instrumen.


In [ ]:
daily_agg = df_hourly.groupby(df_hourly['timestamp'].dt.date).agg(
    year                 = ('year',                  'first'),
    month                = ('month',                 'first'),
    day                  = ('day',                   'first'),
    suhu_rerata_c        = ('suhu_bola_kering_c',    'mean'),
    suhu_max_obs_c       = ('suhu_bola_kering_c',    'max'),
    suhu_min_obs_c       = ('suhu_bola_kering_c',    'min'),
    titik_embun_rerata_c = ('suhu_titik_embun_c',    'mean'),
    kelembaban_rerata_pct= ('kelembaban_relatif_pct','mean'),
    tekanan_qff_rerata_mb= ('tekanan_qff_mb',        'mean'),
    tekanan_qfe_rerata_mb= ('tekanan_qfe_mb',        'mean'),
    curah_hujan_total_mm = ('rainfall_mm',           'sum'),
    jam_hujan            = ('rainfall_mm',           lambda x: (x > 0).sum()),
    visibility_rerata_km = ('visibility_km',         'mean'),
    tutupan_awan_rerata  = ('tutupan_awan_oktas',    'mean'),
    kec_angin_max_ms     = ('wind_speed_ms',         'max'),
    kec_angin_rerata_ms  = ('wind_speed_ms',         'mean'),
    n_observasi          = ('suhu_bola_kering_c',    'count'),
).reset_index().rename(columns={'timestamp': 'date'})

daily_agg['date'] = pd.to_datetime(daily_agg['date'])


In [ ]:
df_12 = df[df['hour'] == 12][['date', 'TEMP MAX C TXTXTX', 'TEMP MIN C TNTNTN']].copy()
df_12['date'] = pd.to_datetime(df_12['date'])
df_12 = df_12.rename(columns={
    'TEMP MAX C TXTXTX': 'suhu_max_tercatat_c',
    'TEMP MIN C TNTNTN': 'suhu_min_tercatat_c',
})

df_00 = df[df['hour'] == 0][['date', 'SUNSHINE H SSS', 'RAINFALL 24H RRRR']].copy()
df_00['date'] = pd.to_datetime(df_00['date'])
df_00['RAINFALL 24H RRRR'] = df_00['RAINFALL 24H RRRR'].replace(8888, np.nan)
df_00 = df_00.rename(columns={
    'SUNSHINE H SSS'   : 'lama_penyinaran_jam',
    'RAINFALL 24H RRRR': 'curah_hujan_24h_mm',
})

df_daily = daily_agg.merge(df_12, on='date', how='left')
df_daily = df_daily.merge(df_00, on='date', how='left')

numeric_cols = df_daily.select_dtypes(include=[np.number]).columns
df_daily[numeric_cols] = df_daily[numeric_cols].round(2)

print(f'Tabel harian: {df_daily.shape[0]:,} baris × {df_daily.shape[1]} kolom')


## 6. Statistik Deskriptif Hasil Cleaning

In [ ]:
print('clean_hourly.csv')
display(df_hourly.describe().round(3))


In [ ]:
print('clean_daily.csv')
display(df_daily.describe().round(3))


In [ ]:
print('Missing values — clean_hourly.csv')
h_miss = (df_hourly.isnull().sum() / len(df_hourly) * 100).round(2)
display(pd.DataFrame({'missing_%': h_miss})[h_miss > 0])

print('\nMissing values — clean_daily.csv')
d_miss = (df_daily.isnull().sum() / len(df_daily) * 100).round(2)
display(pd.DataFrame({'missing_%': d_miss})[d_miss > 0])


## 7. Ekspor

In [ ]:
df_hourly.to_csv(HOURLY_OUT, index=False, encoding='utf-8-sig')
df_daily.to_csv(DAILY_OUT,   index=False, encoding='utf-8-sig')

for path, df_ in [(HOURLY_OUT, df_hourly), (DAILY_OUT, df_daily)]:
    size = os.path.getsize(path) / 1024
    print(f'{os.path.basename(path):<25} {df_.shape[0]:>6,} baris × {df_.shape[1]} kolom  |  {size:.0f} KB')


In [ ]:
print('clean_hourly.csv — 5 baris pertama')
display(df_hourly.head())


In [ ]:
print('clean_daily.csv — 5 baris pertama')
display(df_daily.head())


---
## Referensi

Buck, A. L. (1981). New equations for computing vapor pressure and enhancement factor. *Journal of Applied Meteorology*, *20*(12), 1527–1532. https://doi.org/10.1175/1520-0450(1981)020

Lawrence, M. G. (2005). The relationship between relative humidity and the dewpoint temperature in moist air: A simple conversion and applications. *Bulletin of the American Meteorological Society*, *86*(2), 225–233. https://doi.org/10.1175/BAMS-86-2-225

Mardia, K. V., & Jupp, P. E. (2000). *Directional statistics*. John Wiley & Sons. https://doi.org/10.1002/9780470316979

Moritz, S., & Bartz-Beielstein, T. (2017). imputeTS: Time series missing value imputation in R. *The R Journal*, *9*(1), 207–218. https://doi.org/10.32614/RJ-2017-009

Sanchez-Lorenzo, A., Calbó, J., & Wild, M. (2013). Global and diffuse solar radiation in Spain: Building a homogeneous dataset changes and trends. *Global and Planetary Change*, *100*, 343–352. https://doi.org/10.1016/j.gloplacha.2012.11.010

Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, *59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10

WMO. (2017). *International cloud atlas: Manual on the observation of clouds and other meteors* (WMO-No. 407). World Meteorological Organization. https://cloudatlas.wmo.int

WMO. (2018). *Guide to meteorological instruments and methods of observation* (WMO-No. 8, 2018 ed.). World Meteorological Organization. https://library.wmo.int/records/item/41650-guide-to-meteorological-instruments-and-methods-of-observation

WMO. (2019). *Manual on codes — International codes, Volume I.1* (WMO-No. 306). World Meteorological Organization. https://library.wmo.int/records/item/35713-manual-on-codes-volume-i-1

Zahumenský, I. (2004). *Guidelines on quality control procedures for data from automatic weather stations*. World Meteorological Organization, CIMO.
